In [ ]:
library(dplyr)
library(tidyr)
library(purrr)
library(stringr)

#
source("NewSiteDataLoader.R")
source("data_loader.R")
source("meta.R")
source("recalMNE.R")
source("select_data.R")
ls()

In [ ]:
# Create NewSiteData_sum and NewSiteData_ind datasets
n_samples <- 1
NewSiteData_ind <- generate_dataset(n_samples)
NewSiteData_ind$number_of_individuals <- 24
NewSiteData_sum <- add_element_count(manipulate_rows(NewSiteData_ind))

NewSiteData_ind <- generate_dataset(n_samples)
NewSiteData_ind$number_of_individuals <- 7
NewSiteData_ind <- add_element_count(manipulate_rows(NewSiteData_ind))

# Save NewSiteData_sum and NewSiteData_ind as .rd files
save(NewSiteData_sum, file = "NewSiteData_sum.rda")
save(NewSiteData_ind, file = "NewSiteData_ind.rda")

print("### SUM ###")
print(colnames(NewSiteData_sum))

In [ ]:
# Load the datasets
load("HomininPartAbundance_small.rda")
load("HomininPartAbundance_large.rda")
#load("HomininPartAbundance_XL.rda")
load("NewSiteData_sum.rda")
load("NewSiteData_ind.rda")

# If they are already tibbles, you don't need to convert them again.
# You can just start using them as they are
#print(HomininPartAbundance_small)
print("### SUM ###")
print(NewSiteData_sum)
print("### IND ###")
print(NewSiteData_ind)

In [ ]:
same_data <- select_data(
  data = HomininPartAbundance_large,
  #time_range =  "Medieval Period",
  #geological_context = "Cave",
  #number_of_individuals = 70,
  #age = "Adult (17+ years)",
  #gender = "Female",
  accumulation_cause = "Cannibalism",
  #bone_parts = c("Skull", "Arms"),
  #bone_level = c(2,1,3,4,2,1),
  #site_name = c("Qafzeh", "Regourdou"),
  #sum = 1,
  #hominin_species = "Australopithecus africanus"
)
same_data

In [ ]:
recalc_data <-recal_mne_selected_data(same_data)

In [ ]:
recalc_data[615:635,10:14]



In [ ]:
convertMNEtoVariables <- function(MNE, skeleton, individuals = NULL) {

  if (length(MNE) != nrow(skeleton)) {
    stop("Length of MNE vector must match the number of skeletal elements")
  }

  # Convert MNE to a data frame if it's not already
  if (!is.data.frame(MNE)) {
    MNE <- as.data.frame(MNE)
  }

  Element <- skeleton$element

  # Calculate MNI and Individuals if not provided
  if (is.null(individuals)) {
    MNI <- ceiling(MNE$MNE / skeleton$count)
    individuals <- max(MNI)
  }

  # Compute Abundance for each part
  abundance <- MNE$MNE / (skeleton$count * individuals)

  # Compute Mean Abundance (MA)
  ma <- sum(MNE$MNE, na.rm = TRUE) / (sum(skeleton$count, na.rm = TRUE) * individuals)

  # Compute Mean Part Abundance (MPA)
  mpa <- sum(abundance, na.rm = TRUE) / length(abundance)

  # Additional metrics calculations
  MAU <- abundance
  MNAU <- abundance
  PMAU <- MAU / max(MNE$MNE / skeleton$count)
  PMNAU <- MNAU / individuals
  ReMNAU <- MNAU / sum(MNAU, na.rm = TRUE)

  # Calculate Accumulative ReMNAU
  AcReMNAU <- cumsum(ReMNAU)

  # Bone level vector calculation using summarize_bone_sections
  #bone_info_df <- data.frame(elements = skeleton$Element, counts = MNE$MNE)
  #bone_level <- c(1,1,2,2,2,2,2,2,2,2,2,2)

  # Return a list of all calculated metrics
  return(list(
    MNE = as.vector(MNE$MNE),
    Elements = Element,
    abundance = abundance,
    MNI = individuals,
    MAU = MAU,
    MNAU = MNAU,
    PercentMAU = PMAU,
    PercentMNAU = PMNAU,
    ReMNAU = ReMNAU,
    AcReMNAU = AcReMNAU,
    MeanAbundance = ma,
    MeanPartAbundance = mpa
    #bone_level = bone_level
  ))
}

In [ ]:
skeleton <- get_bones_dataframe(recalc_data$bone.level[[1]])

In [ ]:

Burial <- convertMNEtoVariables(recalc_data[1,]$count[[1]],skeleton ,recalc_data[1,]$number_of_individuals[[1]])

In [ ]:
Burial

In [ ]:
library(dplyr)
library(purrr)

# Assuming your tibble is called 'recalc_data'
# Step 1: Create a helper function that applies `convertMNEtoVariables` to each row.
apply_conversion <- function(count, bone_level, number_of_individuals) {
  skeleton <- get_bones_dataframe(bone_level)
  result <- convertMNEtoVariables(count, skeleton, number_of_individuals)
  
  # Return the list of computed values you want to add to the dataframe
  return(result)
}

# Step 2: Use `pmap` to apply the function to each row and add computed values as new columns
# Here we assume your 'count', 'bone.level', and 'number_of_individuals' columns contain lists, as in your example.
recalc_data <- recalc_data %>%
  mutate(
    results = pmap(
      list(count = count, bone_level = bone.level, number_of_individuals = number_of_individuals),
      apply_conversion
    )
  )

# Step 3: Extract specific metrics from the results and add them as new columns
recalc_data <- recalc_data %>%
  mutate(
    MNE = map(results, ~.x$MNE),
    Elements = map(results, ~.x$Elements),
    abundance = map(results, ~.x$abundance),
    MNI = map(results, ~.x$MNI),
    MAU = map(results, ~.x$MAU),
    PercentMAU = map(results, ~.x$PercentMAU),
    AcReMNAU = map(results, ~.x$AcReMNAU),
    MeanAbundance = map(results, ~.x$MeanAbundance),
    MeanPartAbundance = map(results, ~.x$MeanPartAbundance)
  ) %>%
  select(-results)  # Optionally remove the intermediate 'results' column if you don't need it


In [ ]:
colnames(recalc_data)

In [ ]:
library(ggplot2)

# Plotting MAU
ggplot(recalc_data[10:20], aes(x = element, y = unlist(MAU))) +
  geom_bar(stat = "identity") +
  labs(title = "MAU by Site", x = "Site Name", y = "MAU")

# Plotting PercentMAU
ggplot(recalc_data[10:20], aes(x = element, y = unlist(PercentMAU))) +
  geom_bar(stat = "identity") +
  labs(title = "Percent MAU by Site", x = "Site Name", y = "Percent MAU")

# Plotting AcReMNAU
ggplot(recalc_data[10:20], aes(x = element, y = unlist(AcReMNAU))) +
  geom_line() +
  labs(title = "Accumulative ReMNAU by Site", x = "Site Name", y = "Accumulative ReMNAU")


In [ ]:
install.packages("plotly")


In [ ]:
library(plotly)
library(dplyr)



# Plotly interactive plot with buttons to toggle metrics
fig <- plot_ly()

# Add traces for each metric
# These traces will be controlled by the buttons later
fig <- fig %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$MNE[[1]]),
            mode = 'lines+markers', name = 'MNE', visible = TRUE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$MAU[[1]]),
            mode = 'lines+markers', name = 'MAU', visible = FALSE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$PercentMAU[[1]]),
            mode = 'lines+markers', name = 'PercentMAU', visible = FALSE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$AcReMNAU[[1]]),
            mode = 'lines+markers', name = 'AcReMNAU', visible = FALSE)

# Add buttons for switching between the metrics
fig <- fig %>%
  layout(
    title = "Hominin Bone Metrics by Element",
    xaxis = list(title = "Element"),
    yaxis = list(title = "Value"),
    updatemenus = list(
      list(
        type = "buttons",
        direction = "right",
        xanchor = 'center',
        yanchor = 'top',
        showactive = TRUE,
        buttons = list(
          list(method = "restyle",
               args = list("visible", list(TRUE, FALSE, FALSE, FALSE)),
               label = "MNE"),
          list(method = "restyle",
               args = list("visible", list(FALSE, TRUE, FALSE, FALSE)),
               label = "MAU"),
          list(method = "restyle",
               args = list("visible", list(FALSE, FALSE, TRUE, FALSE)),
               label = "PercentMAU"),
          list(method = "restyle",
               args = list("visible", list(FALSE, FALSE, FALSE, TRUE)),
               label = "AcReMNAU")
        )
      )
    )
  )

# Display the plot
fig


In [ ]:

# Plotly interactive plot with buttons to toggle metrics
fig <- plot_ly()

# Add traces for each metric
# These traces will be controlled by the buttons later
fig <- fig %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$MNE[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'MNE', visible = TRUE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$MAU[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'MAU', visible = FALSE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$PercentMAU[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'PercentMAU', visible = FALSE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$AcReMNAU[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'AcReMNAU', visible = FALSE)

# Add buttons for switching between the metrics
fig <- fig %>%
  layout(
    title = "Hominin Bone Metrics by Element",
    xaxis = list(title = "Element"),
    yaxis = list(title = "Value"),
    updatemenus = list(
      list(
        type = "buttons",
        direction = "right",
        xanchor = 'center',
        yanchor = 'top',
        showactive = TRUE,
        buttons = list(
          list(method = "restyle",
               args = list("visible", list(TRUE, FALSE, FALSE, FALSE)),
               label = "MNE"),
          list(method = "restyle",
               args = list("visible", list(FALSE, TRUE, FALSE, FALSE)),
               label = "MAU"),
          list(method = "restyle",
               args = list("visible", list(FALSE, FALSE, TRUE, FALSE)),
               label = "PercentMAU"),
          list(method = "restyle",
               args = list("visible", list(FALSE, FALSE, FALSE, TRUE)),
               label = "AcReMNAU")
        )
      )
    )
  )

# Display the plot
fig


In [ ]:
# Install necessary packages if you haven't
# install.packages("plotly")
# install.packages("dplyr")
# install.packages("purrr")
#install.packages("IRdisplay")

library(plotly)
library(dplyr)
library(IRdisplay)



# Plotly interactive plot with buttons to toggle metrics
fig <- plot_ly()

# Add traces for each metric
fig <- fig %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$MNE[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'MNE', visible = TRUE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$MAU[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'MAU', visible = FALSE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$PercentMAU[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'PercentMAU', visible = FALSE) %>%
  add_trace(x = unlist(recalc_data$Elements[[1]]), y = unlist(recalc_data$AcReMNAU[[1]]),
            mode = 'lines+markers', type = 'scatter', name = 'AcReMNAU', visible = FALSE)

# Add buttons for switching between the metrics
fig <- fig %>%
  layout(
    title = "Hominin Bone Metrics by Element",
    xaxis = list(title = "Element"),
    yaxis = list(title = "Value"),
    updatemenus = list(
      list(
        type = "buttons",
        direction = "right",
        xanchor = 'center',
        yanchor = 'top',
        showactive = TRUE,
        buttons = list(
          list(method = "restyle",
               args = list("visible", list(TRUE, FALSE, FALSE, FALSE)),
               label = "MNE"),
          list(method = "restyle",
               args = list("visible", list(FALSE, TRUE, FALSE, FALSE)),
               label = "MAU"),
          list(method = "restyle",
               args = list("visible", list(FALSE, FALSE, TRUE, FALSE)),
               label = "PercentMAU"),
          list(method = "restyle",
               args = list("visible", list(FALSE, FALSE, FALSE, TRUE)),
               label = "AcReMNAU")
        )
      )
    )
  )

# Display the plot inline using IRdisplay
IRdisplay::display_html(fig)


In [ ]:
# Simple test plot using plotly
library(plotly)

# Create a simple data frame
test_data <- data.frame(
  x = c(1, 2, 3, 4, 5),
  y = c(10, 11, 12, 13, 14)
)

# Create a simple scatter plot using plotly
test_plot <- plot_ly(data = test_data, x = ~x, y = ~y, type = 'scatter', mode = 'lines+markers')

# Display the plot
IRdisplay::display_html(test_plot)


In [ ]:
# Simple test plot using plotly
library(plotly)

# Create a simple data frame
test_data <- data.frame(
  x = c(1, 2, 3, 4, 5),
  y = c(10, 11, 12, 13, 14)
)

# Create a simple scatter plot using plotly
test_plot <- plot_ly(data = test_data, x = ~x, y = ~y, type = 'scatter', mode = 'lines+markers')

# Display the plot directly
test_plot


In [ ]:
# Check if plotly is correctly loaded and render the simple plot
library(plotly)
plot_ly(x = ~rnorm(100), y = ~rnorm(100), type = "scatter", mode = "markers")


In [ ]:
# Load necessary libraries
library(ggplot2)
library(plotly)

# Create a ggplot2 object
p <- ggplot(mtcars, aes(x = wt, y = mpg)) + geom_point()

# Convert ggplot2 object to a plotly interactive plot
ggplotly(p)


In [3]:
# Install or update necessary packages
install.packages("plotly")
install.packages("IRkernel")
install.packages("repr")
install.packages("htmlwidgets")


Installing package into ‘C:/Users/Usuario/AppData/Local/R/win-library/4.3’
(as ‘lib’ is unspecified)
Installing package into ‘C:/Users/Usuario/AppData/Local/R/win-library/4.3’
(as ‘lib’ is unspecified)


trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.3/plotly_4.10.4.zip'


Content type 'application/zip' length 3258553 bytes (3.1 MB)


downloaded 3.1 MB



package ‘plotly’ successfully unpacked and MD5 sums checked
package ‘plotly’ successfully unpacked and MD5 sums checked


...

Content type 'application/zip' length 812181 bytes (793 KB)


downloaded 793 KB



package ‘htmltools’ successfully unpacked and MD5 sums checked
package ‘htmltools’ successfully unpacked and MD5 sums checked


package ‘htmlwidgets’ successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Usuario\AppData\Local\Temp\Rtmpesbw13\downloaded_packages
package ‘htmlwidgets’ successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Usuario\AppData\Local\Temp\Rtmpesbw13\downloaded_packages


Warning message:
In file.copy(savedcopy, lib, recursive = TRUE) :
  problem copying C:\Users\Usuario\AppData\Local\R\win-library\4.3\00LOCK\htmltools\libs\x64\htmltools.dll to C:\Users\Usuario\AppData\Local\R\win-library\4.3\htmltools\libs\x64\htmltools.dll: Permission denied


In [ ]:
%inline%
library(plotly)
library(repr) # This ensures correct size handling in Jupyter

# Create a simple scatter plot
test_plot <- plot_ly(x = c(1, 2, 3, 4, 5), y = c(10, 11, 12, 13, 14), 
                     type = 'scatter', mode = 'lines+markers')

# Ensure the plot is sized correctly for Jupyter
options(repr.plot.width = 8, repr.plot.height = 5)

# Display the plot inline
test_plot


In [ ]:
IRkernel::installspec()  # Install R kernel for Jupyter


In [ ]:
library(htmlwidgets)
library(plotly)

# Simple test plot using plotly
test_plot <- plot_ly(x = c(1, 2, 3, 4, 5), y = c(10, 11, 12, 13, 14), type = 'scatter', mode = 'lines+markers')

# Save plot as an HTML file and open it in your browser
saveWidget(test_plot, "test_plot.html")

# Manually open the test_plot.html file in your browser to verify the plot


In [1]:
install.packages("ggplot2")

Installing package into ‘C:/Users/Usuario/AppData/Local/R/win-library/4.3’
(as ‘lib’ is unspecified)
Installing package into ‘C:/Users/Usuario/AppData/Local/R/win-library/4.3’
(as ‘lib’ is unspecified)


trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.3/ggplot2_3.5.1.zip'


Content type 'application/zip' length 4952315 bytes (4.7 MB)


downloaded 4.7 MB



package ‘ggplot2’ successfully unpacked and MD5 sums checked
package ‘ggplot2’ successfully unpacked and MD5 sums checked



The downloaded binary packages are in
	C:\Users\Usuario\AppData\Local\Temp\Rtmpesbw13\downloaded_packages


The downloaded binary packages are in
	C:\Users\Usuario\AppData\Local\Temp\Rtmpesbw13\downloaded_packages


In [5]:
library(plotly)
fig <- plot_ly(midwest, x = ~percollege, color = ~state, type = "box")
fig

Warning message:
package ‘plotly’ was built under R version 4.3.3 


: Error:
! package or namespace load failed for ‘plotly’ in loadNamespace(i, c(lib.loc, .libPaths()), versionCheck = vI[[i]]):
 namespace ‘htmltools’ 0.5.6.1 is already loaded, but >= 0.5.7 is required

In [4]:
install.packages("htmltools")


Installing package into ‘C:/Users/Usuario/AppData/Local/R/win-library/4.3’
(as ‘lib’ is unspecified)
Installing package into ‘C:/Users/Usuario/AppData/Local/R/win-library/4.3’
(as ‘lib’ is unspecified)


trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.3/htmltools_0.5.8.1.zip'


Content type 'application/zip' length 359060 bytes (350 KB)
downloaded 350 KB



package ‘htmltools’ successfully unpacked and MD5 sums checked
package ‘htmltools’ successfully unpacked and MD5 sums checked



The downloaded binary packages are in
	C:\Users\Usuario\AppData\Local\Temp\Rtmpesbw13\downloaded_packages

The downloaded binary packages are in
	C:\Users\Usuario\AppData\Local\Temp\Rtmpesbw13\downloaded_packages

Warning message:
In file.copy(savedcopy, lib, recursive = TRUE) :
  problem copying C:\Users\Usuario\AppData\Local\R\win-library\4.3\00LOCK\htmltools\libs\x64\htmltools.dll to C:\Users\Usuario\AppData\Local\R\win-library\4.3\htmltools\libs\x64\htmltools.dll: Permission denied


In [6]:
R --version


: Error:
! object 'R' not found